## StableCodec LoViF finetune — offline Kaggle run

Copies the `tonyironman099/stablecodec-main-train` repo into `/kaggle/working`, installs the
needed packages fully offline (auto-discovering any attached `wheels/` dataset, same pattern
as `lovif-training.ipynb`), then launches `src/finetune_lovif.py`.

Attach these datasets before running:
- `tonyironman099/stablecodec-main-train` (the repo)
- `tonyironman099/dpt099-stablecodec-wheels` (pip packages)
- `ahnaftahmid24/lovif-aeic-offline` (sd-turbo weights, + optional dinov2/clip for GAN/CLIP loss terms)
- `mehedi052/stablecodec-checkpoints` (elic_official.pth, stablecodec_base.pkl)
- `tonyironman099/lovif-2026-image-compression` (dataset_train / dataset_val)

In [ ]:
import shutil

src = "/kaggle/input/datasets/tonyironman099/stablecodec-main-train"
dst = "/kaggle/working/StableCodec-main"

shutil.copytree(src, dst, dirs_exist_ok=True)
print("Copied successfully!")

In [ ]:
# =============================================================================
# Offline package install -- same approach as lovif-training.ipynb's setup_env():
# auto-discover any attached `wheels/` dataset (no hardcoded dataset slug), install
# everything via --no-index --find-links --no-deps (Kaggle's base image already has
# every transitive dependency: numpy, pillow, requests, tqdm, pyyaml, scipy, etc.).
#
# compressai special-case: in the dpt099-stablecodec-wheels dataset, compressai==1.2.6
# has no prebuilt wheel for this platform -- pip's `download` step fell back to a
# source sdist, which got auto-extracted into a loose `compressai-1.2.6/compressai-1.2.6/`
# folder instead of staying a .tar.gz. --find-links only recognizes actual archive
# files sitting directly in a directory, not extracted subfolders, so a plain
# --find-links install reports "no versions found" for it even though the source is
# right there. Fix: if a normal compressai install fails, fall back to installing
# directly from that extracted source directory (pip can build from any local folder
# with setup.py/pyproject.toml) -- still fully offline, no internet hit either way.
# =============================================================================
import subprocess, sys, importlib.util
from pathlib import Path

def pip(*args, check=False):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print("\n$ " + " ".join(cmd))
    return subprocess.run(cmd, check=check)

def find_wheels_dirs():
    root = Path("/kaggle/input")
    if not root.exists():
        return []
    found = []
    for d in root.rglob("wheels"):
        if d.is_dir() and any(d.glob("*.whl")):
            found.append(d)
    return list(dict.fromkeys(found))

packages = [
    "gdown", "huggingface_hub", "diffusers", "transformers", "accelerate",
    "peft", "omegaconf", "einops", "timm", "compressai", "open-clip-torch",
    "lpips", "DISTS_pytorch", "pytorch_msssim", "matplotlib",
]

wheels_dirs = find_wheels_dirs()
if not wheels_dirs:
    raise RuntimeError("No wheels/ dataset found under /kaggle/input -- attach dpt099-stablecodec-wheels.")
print("Found wheelhouse(s):", [str(d) for d in wheels_dirs])

find_links = []
for d in wheels_dirs:
    find_links += ["--find-links", str(d)]

pip("--no-index", *find_links, "--no-deps", *packages)

# compressai fallback: only needed if the --find-links install above couldn't find an archive for it.
if importlib.util.find_spec("compressai") is None:
    print("\ncompressai not found via --find-links -- trying extracted-source-dir fallback...")
    for d in wheels_dirs:
        for src_dir in d.glob("compressai-*/compressai-*"):
            if (src_dir / "setup.py").exists() or (src_dir / "pyproject.toml").exists():
                pip("--no-deps", str(src_dir))
                break
        if importlib.util.find_spec("compressai") is not None:
            break

missing = [m for m in ("lpips", "DISTS_pytorch", "compressai", "pytorch_msssim", "transformers", "diffusers", "peft", "accelerate")
           if importlib.util.find_spec(m) is None]
if missing:
    raise RuntimeError(f"Required modules still missing after install: {missing}. Check the wheels/ dataset for these.")

import torch
print("\ntorch:", torch.__version__, "cuda:", torch.cuda.is_available())
print("All required packages installed OK.")

In [ ]:
# StableCodec.py / latent_codec.py do `sys.path.append("..")` (CWD-relative, not
# script-relative) to import the ELIC package one level above src/. That only
# resolves correctly if the CWD is the src/ dir itself when python starts --
# otherwise `..` points at the wrong place and `ELIC` is never found. %cd here
# instead of leaving CWD at /kaggle/working (the notebook default).
%cd /kaggle/working/StableCodec-main/StableCodec-main/src

In [ ]:
!python finetune_lovif.py \
    --sd_path    /kaggle/input/datasets/ahnaftahmid24/lovif-aeic-offline/results/lovif_aeic_offline/sd-turbo \
    --elic_path  /kaggle/input/datasets/mehedi052/stablecodec-checkpoints/elic_official.pth \
    --codec_path /kaggle/input/datasets/mehedi052/stablecodec-checkpoints/stablecodec_base.pkl \
    --train_dirs /kaggle/input/datasets/tonyironman099/lovif-2026-image-compression/dataset_train \
                 /kaggle/input/datasets/tonyironman099/lovif-2026-image-compression/dataset_val \
    --val_dir    /kaggle/input/datasets/tonyironman099/lovif-2026-image-compression/dataset_val \
    --lambda_rate 24 --output_dir /kaggle/working/sc_ft24 \
    --train_patch_size 384 \
    --max_train_steps 50 \
    --checkpointing_steps 500 --keep_last 3 \
    --resume_steps 500 --eval_freq 500 --plot_freq 500 --log_freq 50